# AMLAAAK — prepare the complete pinned historical Divar archive

**Run this in Google Colab, not on your laptop.** Choose **Runtime → Run all**.
No Python editing is needed. A CPU runtime is sufficient; keep the tab connected.
Processing and final read-back may take a substantial time. If Colab disconnects,
no completeness or upload success is implied; use a fresh runtime and run again.

The pinned source currently contains real_estate_ads.csv, not Parquet. It is read with Hugging Face Datasets streaming from the Colab cache; Parquet output is generated here.

This notebook is self-contained and embeds the reviewed repository preparation
code. It reads every CSV/Parquet source partition at the immutable source revision, uses
disk-backed staging and bounded batches, strips personal/free-text fields, and
prepares residential fixed-price **sales only**. Rental rows are counted and
excluded; their prices are never mixed with sales. It does **not** train or change
the existing Random Forest. Source currency units remain unverified.

Nothing is uploaded until you review the results, acknowledge the source ODbL
license, and type the exact final confirmation. Only the existing **private**
`thevel0per/AMLAAAK-Real-Estate-Data` repository can be targeted. You will need
a Hugging Face write token for that dataset (never put it into a code cell).


In [ ]:
%pip -q install pyarrow==23.0.1 pydantic==2.14.3 huggingface_hub==0.36.2 datasets==4.6.1


In [ ]:
# Embedded source: generated from the reviewed AMLAAAK preparation modules.
import json, sys
from pathlib import Path
WORK=Path('/content/amlaaak-preparation')
WORK.mkdir(parents=True,exist_ok=True)
PACKAGE=WORK/'amlaaak_colab'
PACKAGE.mkdir(exist_ok=True)
(PACKAGE/'__init__.py').write_text('',encoding='utf-8')

EMBEDDED_SOURCES={'normalization': '"""Canonical stateless Divar parsing shared by serving and offline training."""\nimport math\nimport re\nfrom datetime import datetime\n\nNUMERIC = [\'building_size\', \'rooms_count\', \'total_floors_count\', \'floor\', \'construction_year\']\nAMENITIES = [\'has_parking\', \'has_elevator\', \'has_balcony\', \'has_warehouse\']\nCATEGORICAL = [\'city_slug\', \'cat3_slug\']\nFEATURES = NUMERIC + AMENITIES + CATEGORICAL\nWORDS = dict(zip([\'صفر\',\'یک\',\'دو\',\'سه\',\'چهار\',\'پنج\',\'شش\',\'هفت\',\'هشت\',\'نه\',\'ده\',\'یازده\',\'دوازده\',\'سیزده\',\'چهارده\',\'پانزده\',\'شانزده\',\'هفده\',\'هجده\',\'نوزده\',\'بیست\'], range(21)))\nWORDS.update({\'بدون اتاق\': 0, \'همکف\': 0, \'زیرزمین\': -1})\nDIGITS = str.maketrans(\'۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩يك\', \'01234567890123456789یک\')\nNAN = float(\'nan\')\n\n\ndef text(value):\n    if value is None or (isinstance(value, float) and not math.isfinite(value)):\n        return \'\'\n    result = str(value).translate(DIGITS).strip().lower()\n    return \'\' if result in (\'nan\', \'null\', \'none\', \'<na>\', \'nat\') else result\n\n\ndef number(value):\n    if isinstance(value,(int,float)) and not isinstance(value,bool):\n        result=float(value)\n        return result if math.isfinite(result) else NAN\n    value = text(value)\n    if value in WORDS:\n        return float(WORDS[value])\n    value = value.replace(\'٬\', \',\').replace(\'٫\', \'.\')\n    if not re.fullmatch(r\'[+-]?(?:\\d{1,3}(?:,\\d{3})+|\\d+)(?:\\.\\d+)?(?:[eE][+-]?\\d+)?\', value):\n        return NAN\n    result = float(value.replace(\',\', \'\'))\n    return result if math.isfinite(result) else NAN\n\n\ndef boolean(value):\n    return {\'true\': 1., \'false\': 0., \'1\': 1., \'0\': 0., \'1.0\': 1., \'0.0\': 0.}.get(text(value), NAN)\n\n\ndef year(value):\n    result = number(value)\n    return result if 1200 <= result <= 1500 and result.is_integer() else NAN\n\n\ndef month(value):\n    value = text(value)[:7]\n    try:\n        return datetime.strptime(value, \'%Y-%m\').strftime(\'%Y-%m\')\n    except ValueError:\n        return None\n\n\ndef normalize_sale(row):\n    """Return safe modeled/context fields or None. No personal text or coordinates."""\n    if (text(row.get(\'cat2_slug\')) != \'residential-sell\'\n            or text(row.get(\'cat3_slug\')) not in (\'apartment-sell\', \'house-villa-sell\')\n            or text(row.get(\'price_mode\')) != \'مقطوع\'):\n        return None\n    price, size = number(row.get(\'price_value\')), number(row.get(\'building_size\'))\n    if not (price > 0 and size > 0):\n        return None\n    result = {}\n    for col in NUMERIC:\n        value = year(row.get(col)) if col == \'construction_year\' else number(row.get(col))\n        if col != \'construction_year\' and value < (-5 if col == \'floor\' else 0):\n            value = NAN\n        result[col] = value if math.isfinite(value) else None\n    for col in AMENITIES:\n        value = boolean(row.get(col))\n        result[col] = bool(value) if math.isfinite(value) else None\n    for col in CATEGORICAL:\n        result[col] = text(row.get(col)) or None\n    result.update(price_value=price, neighborhood_slug=text(row.get(\'neighborhood_slug\')) or None,\n                  created_at_month=month(row.get(\'created_at_month\')))\n    return result\n', 'schemas': "from typing import Literal\nfrom pydantic import BaseModel, ConfigDict, Field, model_validator\n\n\nclass Contract(BaseModel):\n    model_config = ConfigDict(extra='forbid', allow_inf_nan=False)\n\n\nclass Provenance(Contract):\n    kind: Literal['illustrative', 'divar']\n    source_url: str\n    source_sha256: str | None\n    revision: str\n    input_rows: int\n    eligible_rows: int\n    units: str = 'Original price_value units (currency unverified)'\n    note: str\n    provider: str = 'snapshot'\n    coverage: 'Coverage | None' = None\n    fetched_at: str | None = None\n    period_start: str | None = None\n    period_end: str | None = None\n\n\nclass Coverage(Contract):\n    records_examined: int\n    normalized_records: int\n    eligible_records: int\n    upstream_matching_rows: int | None = None\n    exhaustive: bool\n    bounded: bool\n    sampled: bool = False\n    statistics_full_population: bool\n    remote_filter_supported: bool\n    remote_sort_supported: bool = False\n    sorting_scope: str = 'retrieved_records'\n    method: str\n    limitations: list[str] = Field(default_factory=list)\n    source_processing_complete: bool | None = None\n\n\nclass Property(Contract):\n    id: str\n    city_slug: str\n    cat3_slug: Literal['apartment-sell', 'house-villa-sell']\n    building_size: float = Field(gt=0)\n    price_value: float = Field(gt=0)\n    price_per_sqm: float = Field(gt=0)\n    rooms_count: float | None\n    floor: float | None\n    total_floors_count: float | None\n    construction_year: float | None\n    has_parking: bool | None\n    has_elevator: bool | None\n    has_balcony: bool | None\n    has_warehouse: bool | None\n    neighborhood_slug: str | None = Field(default=None, exclude_if=lambda v: v is None)\n    created_at_month: str | None = Field(default=None, exclude_if=lambda v: v is None)\n    source_identity: str | None = Field(default=None, exclude_if=lambda v: v is None)\n    source_row_index: int | None = Field(default=None, exclude_if=lambda v: v is None)\n    source_revision: str | None = Field(default=None, exclude_if=lambda v: v is None)\n\n\nclass Snapshot(Contract):\n    provenance: Provenance\n    properties: list[Property]\n\n\nclass Filters(Contract):\n    city: str | None = Field(default=None, max_length=80)\n    neighborhood: str | None = Field(default=None, max_length=100)\n    bedrooms_min: int | None = Field(default=None, ge=0, le=20)\n    bedrooms_max: int | None = Field(default=None, ge=0, le=20)\n    property_type: Literal['apartment-sell', 'house-villa-sell'] | None = None\n    area_min: float | None = Field(default=None, ge=0)\n    area_max: float | None = Field(default=None, ge=0)\n    price_min: float | None = Field(default=None, ge=0)\n    price_max: float | None = Field(default=None, ge=0)\n    year_min: int | None = Field(default=None, ge=1200, le=1500)\n    year_max: int | None = Field(default=None, ge=1200, le=1500)\n    search: str = Field(default='', max_length=80)\n\n    @model_validator(mode='after')\n    def valid_ranges(self):\n        for name in ('area', 'price', 'year', 'bedrooms'):\n            low, high = getattr(self, name + '_min'), getattr(self, name + '_max')\n            if low is not None and high is not None and low > high:\n                raise ValueError(f'{name}_min must not exceed {name}_max')\n        return self\n\n\nclass ModelStatus(Contract):\n    available: bool\n    mode: Literal['unavailable', 'illustrative', 'trained']\n    name: str = 'Random Forest'\n    version: str | None = None\n    message: str\n    source_revision: str | None = None\n\n\nclass PropertyFilters(Filters):\n    page: int = Field(default=1, ge=1)\n    page_size: int = Field(default=10, ge=1, le=100)\n    sort: Literal['price_value', 'building_size', 'city_slug', 'price_per_sqm'] = 'price_value'\n    order: Literal['asc', 'desc'] = 'desc'\n\n\nclass Summary(Contract):\n    count: int\n    median_price: float | None\n    average_price: float | None\n    median_price_per_sqm: float | None\n    cities_count: int\n    available_cities: list[str]\n    provenance: Provenance\n    model: ModelStatus\n\n\nclass City(Contract):\n    city: str\n    count: int\n    median_price: float\n    median_price_per_sqm: float\n    latitude: float | None\n    longitude: float | None\n    coverage: Coverage | None = None\n    source_revision: str | None = None\n\n\nclass Bin(Contract):\n    label: str\n    lower: float\n    upper: float\n    count: int\n\n\nclass Count(Contract):\n    label: str\n    count: int\n\n\nclass Amenity(Contract):\n    label: str\n    present: int\n    known: int\n    percent: float | None\n\n\nclass Distributions(Contract):\n    price: list[Bin]\n    area: list[Bin]\n    year: list[Bin]\n    property_types: list[Count]\n    amenities: list[Amenity]\n    price_area: list[dict[str, float | str]]\n    missing_years: int\n    coverage: Coverage | None = None\n\n\nclass PropertyPage(Contract):\n    items: list[Property]\n    total: int\n    page: int\n    page_size: int\n    coverage: Coverage | None = None\n    provenance: Provenance | None = None\n\n\nclass Scores(Contract):\n    mae: float\n    rmse: float\n    r2: float\n\n\nclass Metrics(Contract):\n    status: ModelStatus\n    scores: dict[str, Scores] | None = None\n    train_rows: int | None = None\n    test_rows: int | None = None\n    input_rows: int | None = None\n    eligible_rows: int | None = None\n    seed: int | None = None\n    test_size: float | None = None\n    features: list[str] = Field(default_factory=list)\n    source_sha256: str | None = None\n    versions: dict[str, str] = Field(default_factory=dict)\n    residuals_available: bool = False\n    validation_rows: int | None = None\n    validation_metrics: Scores | None = None\n    split_method: str | None = None\n    source_revision: str | None = None\n    city_metrics: dict = Field(default_factory=dict)\n\n\nclass PredictionInput(Contract):\n    city_slug: str = Field(min_length=1, max_length=80, pattern=r'^[a-z0-9-]+$')\n    neighborhood_slug: str | None = Field(default=None, max_length=100)\n    cat3_slug: Literal['apartment-sell', 'house-villa-sell']\n    building_size: float = Field(gt=0, le=10000)\n    rooms_count: int = Field(ge=0, le=20)\n    floor: int = Field(ge=-5, le=100)\n    total_floors_count: int = Field(ge=1, le=100)\n    construction_year: int = Field(ge=1200, le=1500)\n    has_parking: bool\n    has_elevator: bool\n    has_balcony: bool\n    has_warehouse: bool\n\n    @model_validator(mode='after')\n    def floor_in_building(self):\n        if self.floor > self.total_floors_count:\n            raise ValueError('Floor cannot exceed total floors')\n        return self\n\n\nclass Prediction(Contract):\n    predicted_price: float\n    price_per_sqm: float\n    units: str\n    model: ModelStatus\n    warnings: list[str]\n\n\nclass Health(Contract):\n    status: Literal['ok', 'degraded']\n    dataset_ready: bool\n    model_available: bool\n    provider: str = 'snapshot'\n    upstream_checked: bool = False\n\n\nclass Comparable(Contract):\n    property: Property\n    similarity_score: float = Field(ge=0, le=100)\n    reasons: list[str]\n\n\nclass Valuation(Contract):\n    model: ModelStatus\n    prediction: Prediction | None\n    provenance: Provenance\n    coverage: Coverage\n    comparables: list[Comparable]\n    comparable_median_price: float | None\n    comparable_median_price_per_sqm: float | None\n    prediction_vs_comparable_percent: float | None\n    limitations: list[str]\n    label: str = 'Best matching properties among retrieved historical records.'\n\n\nclass Facets(Contract):\n    cities: list[str]\n    neighborhoods: list[str]\n    provenance: Provenance\n    coverage: Coverage\n\n\nclass ErrorDetail(Contract):\n    code: str\n    message: str\n\n\nclass ErrorResponse(Contract):\n    error: ErrorDetail\n", 'prepared_format': '"""Versioned safe interchange contract shared by Colab and the serving adapter."""\nimport hashlib\nimport json\nimport re\n\nSOURCE_REPO = \'divarofficial/real_estate_ads\'\nSOURCE_REVISION = \'e9446ee97b8e6eb9bd62bdfd7548f1bc8973be85\'\nTARGET_REPO = \'thevel0per/AMLAAAK-Real-Estate-Data\'\nSCHEMA_VERSION = 1\nSORTS = (\'price_value\', \'building_size\', \'city_slug\', \'price_per_sqm\')\nSAFE_FIELDS = {\n    \'id\', \'city_slug\', \'cat3_slug\', \'neighborhood_slug\', \'building_size\', \'price_value\',\n    \'price_per_sqm\', \'rooms_count\', \'floor\', \'total_floors_count\', \'construction_year\',\n    \'has_parking\', \'has_elevator\', \'has_balcony\', \'has_warehouse\', \'created_at_month\',\n    \'source_identity\', \'source_row_index\', \'source_revision\',\n}\n\n\ndef encode(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(\',\', \':\'), allow_nan=False).encode()\n\n\ndef sha256(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with open(path, \'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef group_key(city=None, category=None):\n    return (city or \'*\') + \'|\' + (category or \'*\')\n\n\ndef valid_slug(value, maximum=100):\n    return isinstance(value, str) and len(value) <= maximum and bool(re.fullmatch(r\'[a-z0-9]+(?:-[a-z0-9]+)*\', value))\n\n\ndef validate_record(row):\n    from .schemas import Property\n    if set(row) - SAFE_FIELDS or not valid_slug(row.get(\'city_slug\'), 80):\n        raise ValueError(\'Unexpected/private field or invalid city slug\')\n    if row.get(\'neighborhood_slug\') is not None and not valid_slug(row[\'neighborhood_slug\']):\n        raise ValueError(\'Invalid neighborhood slug\')\n    if not re.fullmatch(r\'[a-f0-9]{16}\', row.get(\'id\', \'\')):\n        raise ValueError(\'Invalid record identity\')\n    return Property.model_validate(row)\n\n\ndef validate_manifest(manifest):\n    if (manifest.get(\'schema_version\') != SCHEMA_VERSION or manifest.get(\'source_repo\') != SOURCE_REPO\n            or manifest.get(\'source_revision\') != SOURCE_REVISION or manifest.get(\'license\') != \'odbl-1.0\'):\n        raise ValueError(\'Unsupported prepared dataset source, schema, or license\')\n    files = manifest.get(\'files\', {})\n    if not files or len(files) > 30000:\n        raise ValueError(\'Invalid manifest file inventory\')\n    for path, info in files.items():\n        if not re.fullmatch(r\'[a-zA-Z0-9_./-]+\', path) or \'..\' in path.split(\'/\') or path.startswith(\'/\'):\n            raise ValueError(\'Unsafe manifest path\')\n        if not re.fullmatch(r\'[a-f0-9]{64}\', info.get(\'sha256\', \'\')) or not 0 < info.get(\'bytes\', 0) <= 32 * 1024 * 1024:\n            raise ValueError(\'Invalid file checksum or size\')\n    if \'catalog.json\' not in files:\n        raise ValueError(\'Catalog missing\')\n    sources=manifest.get(\'source_files\',[])\n    if (not sources or len({p[\'source_path\'] for p in sources})!=len(sources)\n            or sum(p[\'processed_rows\'] for p in sources)!=manifest[\'total_source_rows\']\n            or sum(p[\'expected_rows\'] for p in sources)!=manifest[\'expected_source_rows\']\n            or not 0<manifest[\'total_prepared_rows\']<=manifest[\'total_source_rows\']):\n        raise ValueError(\'Source inventory totals are inconsistent\')\n    if manifest.get(\'full_source_verified\') and (\n        manifest[\'total_source_rows\'] != manifest[\'expected_source_rows\']\n        or not manifest[\'source_files\']\n        or any(p[\'processed_rows\'] != p[\'expected_rows\'] for p in manifest[\'source_files\'])\n    ):\n        raise ValueError(\'Unverified source completeness claim\')\n    return manifest\n\n\ndef validate_catalog(manifest, catalog):\n    """Reject missing indexes and contradictory totals instead of returning empty pages."""\n    files=manifest[\'files\']\n    cities=catalog[\'cities\']\n    groups=catalog[\'groups\']\n    categories=(\'apartment-sell\',\'house-villa-sell\')\n    if (len(cities)!=manifest[\'total_cities\'] or len(cities)!=len(set(cities))\n            or any(not valid_slug(city,80) for city in cities)\n            or groups[\'*|*\'][\'count\']!=manifest[\'total_prepared_rows\']\n            or sum(groups[group_key(city)][\'count\'] for city in cities)!=manifest[\'total_prepared_rows\']):\n        raise ValueError(\'Prepared catalog geographic totals mismatch\')\n    expected={p for p,info in files.items() if info[\'kind\']==\'partition\'}\n    if len(catalog[\'partitions\'])!=len(set(catalog[\'partitions\'])) or set(catalog[\'partitions\'])!=expected:\n        raise ValueError(\'Prepared partition index is incomplete\')\n    counts={}\n    for path in expected:\n        info=files[path]\n        key=group_key(info[\'city\'],info[\'category\'])\n        counts[key]=counts.get(key,0)+info[\'rows\']\n    if sum(counts.values())!=manifest[\'total_prepared_rows\']:\n        raise ValueError(\'Prepared partition count mismatch\')\n    for city in [None]+cities:\n        total=groups[group_key(city)][\'count\']\n        if sum(groups[group_key(city,category)][\'count\'] for category in categories)!=total:\n            raise ValueError(\'Prepared category statistics mismatch\')\n        if city and any(counts.get(group_key(city,category),0)!=groups[group_key(city,category)][\'count\'] for category in categories):\n            raise ValueError(\'Prepared city/category partition totals mismatch\')\n        views=catalog[\'city_views\'][city] if city else catalog[\'views\']\n        for sort in SORTS:\n            paths=views[sort]\n            if len(paths)!=len(set(paths)) or sum(files[p][\'rows\'] for p in paths)!=total:\n                raise ValueError(\'Prepared sorted index is incomplete\')\n            for path in paths:\n                info=files[path]\n                if (info[\'kind\']!=\'view\' or info[\'sort\']!=sort or info.get(\'city\')!=city\n                        or sum(info[\'category_counts\'].values())!=info[\'rows\']):\n                    raise ValueError(\'Prepared sorted index schema mismatch\')\n            for category in categories:\n                if sum(files[p][\'category_counts\'].get(category,0) for p in paths)!=groups[group_key(city,category)][\'count\']:\n                    raise ValueError(\'Prepared sorted category index mismatch\')\n    return catalog\n', 'preparation': '"""Offline, disk-backed preparation. Never imported by the HTTP runtime.\n\nOnly normalized structured sales enter the staging database. SQLite here is an\nephemeral Colab work file, not a deployed database or a service dependency.\n"""\nfrom collections import Counter\nfrom datetime import datetime, timezone\nimport gzip\nimport json\nfrom pathlib import Path\nimport math\nimport sqlite3\n\nfrom .normalization import normalize_sale, text\nfrom .prepared_format import (SOURCE_REPO, SOURCE_REVISION, SCHEMA_VERSION, SORTS,\n                              encode, sha256, file_hash, group_key, valid_slug, validate_record, validate_manifest)\n\nREQUIRED = {\'cat2_slug\', \'cat3_slug\', \'price_mode\', \'price_value\', \'building_size\', \'city_slug\'}\nSOURCE_FIELDS = REQUIRED | {\'rooms_count\', \'floor\', \'total_floors_count\', \'construction_year\',\n    \'has_parking\', \'has_elevator\', \'has_balcony\', \'has_warehouse\', \'neighborhood_slug\', \'created_at_month\'}\n\n\ndef source_batches(path):\n    """Read pinned Parquet directly or the original CSV with HF Datasets streaming."""\n    import pyarrow as pa\n    import pyarrow.parquet as pq\n    path=Path(path)\n    if path.suffix==\'.parquet\':\n        source=pq.ParquetFile(path)\n        columns=set(source.schema_arrow.names)\n        if not REQUIRED<=columns:\n            raise ValueError(\'Source schema missing required fields\')\n        yield from source.iter_batches(batch_size=1024,columns=sorted(SOURCE_FIELDS & columns))\n    elif path.suffix==\'.csv\':\n        import csv\n        from datasets import load_dataset, Features, Value\n        with path.open(encoding=\'utf-8-sig\',newline=\'\') as stream:\n            header=next(csv.reader(stream))\n        if len(set(header))!=len(header) or not REQUIRED<=set(header):\n            raise ValueError(\'Source CSV schema missing or duplicated fields\')\n        source=load_dataset(\'csv\',data_files={\'train\':str(path)},split=\'train\',streaming=True,\n                            features=Features({name:Value(\'string\') for name in header}))\n        source=source.select_columns(sorted(SOURCE_FIELDS & set(header)))\n        for batch in source.iter(batch_size=1024):\n            yield pa.RecordBatch.from_pydict(batch)\n    else:\n        raise ValueError(\'Unsupported source partition format\')\n\n\ndef inspect_source(path):\n    # An independent complete streaming pass verifies readable EOF and source row\n    # counts before the normalization pass; no source record is retained.\n    return sum(batch.num_rows for batch in source_batches(path))\n\n\ndef build_dataset(source_parts, output, *, chunk_rows=2000, progress=print):\n    """Each part: path(local), source_path, sha256, expected_rows from a pinned file.\n\n    All source parts must be supplied. No max-row option is provided. Completeness\n    is relative to this verified inventory, not active listings or the market.\n    """\n    import pyarrow as pa\n    import pyarrow.parquet as pq\n    if not source_parts or len({p[\'source_path\'] for p in source_parts}) != len(source_parts):\n        raise ValueError(\'Expected source partition inventory is empty or duplicated\')\n    if not 1 <= chunk_rows <= 2000:\n        raise ValueError(\'Chunk size must be between 1 and 2000\')\n    output = Path(output)\n    if output.exists() and any(output.iterdir()):\n        raise ValueError(\'Use a new empty output folder; previous data is preserved\')\n    output.mkdir(parents=True, exist_ok=True)\n    stage = output.parent / (output.name + \'-staging.sqlite\')\n    if stage.exists():\n        raise ValueError(\'Staging file exists; choose a fresh output name\')\n    db = sqlite3.connect(stage)\n    db.execute(\'PRAGMA temp_store=FILE\')\n    db.execute(\'PRAGMA cache_size=-8192\')\n    db.execute(\'CREATE TABLE listings (id TEXT PRIMARY KEY, city_slug TEXT, cat3_slug TEXT, neighborhood_slug TEXT, \'\n               \'price_value REAL, building_size REAL, price_per_sqm REAL, payload TEXT) WITHOUT ROWID\')\n    counts, inventory = Counter(), []\n    db.execute(\'CREATE TABLE lookup (id TEXT PRIMARY KEY, path TEXT) WITHOUT ROWID\')\n    files, partitions = {}, []\n\n    def save(path, data, kind, **attributes):\n        target = output / path\n        target.parent.mkdir(parents=True, exist_ok=True)\n        target.write_bytes(data)\n        files[path] = dict(sha256=sha256(data), bytes=len(data), kind=kind, **attributes)\n        return path\n\n    def save_chunk(rows, path, kind, **attrs):\n        records = [json.loads(row[0]) for row in rows]\n        for record in records:\n            validate_record(record)\n        save(path, gzip.compress(encode(records), mtime=0), kind, rows=len(records), **attrs)\n        if kind == \'view\':\n            files[path][\'category_counts\'] = dict(Counter(p[\'cat3_slug\'] for p in records))\n        return records\n\n    try:\n        for part in source_parts:\n            path = Path(part[\'path\'])\n            if file_hash(path) != part[\'sha256\']:\n                raise ValueError(\'Source file hash mismatch\')\n            processed = 0\n            for batch in source_batches(path):\n                for raw in batch.to_pylist():\n                    index = counts[\'source_rows\']\n                    counts[\'source_rows\'] += 1\n                    processed += 1\n                    if text(raw.get(\'cat2_slug\')) == \'residential-rent\':\n                        counts[\'rental_rows_excluded\'] += 1\n                        continue\n                    normalized = normalize_sale(raw)\n                    if normalized is None:\n                        counts[\'other_ineligible_rows\'] += 1\n                        continue\n                    if not math.isfinite(normalized[\'price_value\']/normalized[\'building_size\']):\n                        counts[\'invalid_price_per_sqm\'] += 1\n                        continue\n                    if not valid_slug(normalized[\'city_slug\'], 80):\n                        counts[\'invalid_city_rows\'] += 1\n                        continue\n                    if normalized[\'neighborhood_slug\'] and not valid_slug(normalized[\'neighborhood_slug\']):\n                        normalized[\'neighborhood_slug\'] = None\n                        counts[\'invalid_neighborhood_values\'] += 1\n                    # Safe-field equivalence, not a claim that different homes are the same.\n                    signature = encode(normalized)\n                    identity = sha256(signature)[:16]\n                    counts[\'eligible_before_dedup\'] += 1\n                    normalized.update(id=identity, price_per_sqm=normalized[\'price_value\']/normalized[\'building_size\'],\n                        source_revision=SOURCE_REVISION, source_row_index=index,\n                        source_identity=f\'hf://datasets/{SOURCE_REPO}@{SOURCE_REVISION}/{part["source_path"]}#row={processed-1}\')\n                    validate_record(normalized)\n                    inserted = db.execute(\'INSERT OR IGNORE INTO listings VALUES (?,?,?,?,?,?,?,?)\',\n                        (identity, normalized[\'city_slug\'], normalized[\'cat3_slug\'], normalized[\'neighborhood_slug\'],\n                         normalized[\'price_value\'], normalized[\'building_size\'], normalized[\'price_per_sqm\'], encode(normalized).decode()))\n                    if inserted.rowcount==0:\n                        existing=json.loads(db.execute(\'SELECT payload FROM listings WHERE id=?\',(identity,)).fetchone()[0])\n                        if {k:existing[k] for k in json.loads(signature)}!=json.loads(signature):\n                            raise ValueError(\'Record identity hash collision; preparation stopped\')\n                db.commit()\n                progress(f\'Processed {counts["source_rows"]:,} source rows\')\n            if processed != part[\'expected_rows\']:\n                raise ValueError(\'Source partition was not processed completely\')\n            inventory.append({k: part[k] for k in (\'source_path\', \'sha256\', \'expected_rows\') } | {\'processed_rows\': processed})\n\n        total = db.execute(\'SELECT count(*) FROM listings\').fetchone()[0]\n        if not total:\n            raise ValueError(\'No eligible prepared sales\')\n        counts[\'duplicates_removed\'] = counts[\'eligible_before_dedup\'] - total\n        db.execute(\'CREATE INDEX geographic ON listings(city_slug,cat3_slug,neighborhood_slug)\')\n        for sort in SORTS:\n            db.execute(f\'CREATE INDEX sort_{sort} ON listings({sort},id)\')\n        cities = [r[0] for r in db.execute(\'SELECT DISTINCT city_slug FROM listings ORDER BY city_slug\')]\n        categories = [\'apartment-sell\', \'house-villa-sell\']\n\n        def clause(city, category):\n            pairs = [(k, v) for k, v in [(\'city_slug\', city), (\'cat3_slug\', category)] if v]\n            return (\' WHERE \' + \' AND \'.join(k+\'=?\' for k,v in pairs) if pairs else \'\'), tuple(v for k,v in pairs)\n\n        def aggregate(city, category):\n            where, params = clause(city, category)\n            n, avg = db.execute(\'SELECT count(*),avg(price_value) FROM listings\'+where, params).fetchone()\n            def median(field):\n                if not n:\n                    return None\n                values = db.execute(f\'SELECT {field} FROM listings{where} ORDER BY {field} LIMIT ? OFFSET ?\',\n                                    params + (2 if n % 2 == 0 else 1, (n-1)//2)).fetchall()\n                return sum(v[0] for v in values)/len(values)\n            # Stream to a tiny SQL aggregate table rather than constructing all Property objects.\n            amenities = []\n            for feature in (\'has_parking\',\'has_elevator\',\'has_balcony\',\'has_warehouse\'):\n                present, known = db.execute(f"SELECT sum(json_extract(payload,\'$.{feature}\')), count(json_extract(payload,\'$.{feature}\')) FROM listings"+where,params).fetchone()\n                present = present or 0\n                amenities.append(dict(label=feature.removeprefix(\'has_\'),present=present,known=known,percent=100*present/known if known else None))\n            def histogram(field):\n                expression = field if field in (\'price_value\',\'building_size\') else f"json_extract(payload,\'$.{field}\')"\n                lo, hi = db.execute(f\'SELECT min({expression}),max({expression}) FROM listings\'+where,params).fetchone()\n                if lo is None:\n                    return []\n                if lo == hi:\n                    count = db.execute(f\'SELECT count({expression}) FROM listings\'+where,params).fetchone()[0]\n                    return [dict(label=f\'{lo:g}\',lower=lo,upper=hi,count=count)]\n                width = (hi-lo)/6\n                buckets = dict(db.execute(f\'SELECT min(5,cast(({expression}-?)/? AS INTEGER)),count(*) FROM listings\'+where+\n                    (\' AND \' if where else \' WHERE \')+f\'{expression} IS NOT NULL GROUP BY 1\', (lo,width)+params).fetchall())\n                return [dict(label=f\'{lo+i*width:.3g}–{lo+(i+1)*width:.3g}\',lower=lo+i*width,upper=lo+(i+1)*width,count=buckets.get(i,0)) for i in range(6)]\n            sample = [json.loads(r[0]) for r in db.execute(\'SELECT payload FROM listings\'+where+\' ORDER BY id LIMIT 20\', params)]\n            city_list = [r[0] for r in db.execute(\'SELECT DISTINCT city_slug FROM listings\'+where+\' ORDER BY city_slug\',params)]\n            neighborhoods = [r[0] for r in db.execute(\'SELECT DISTINCT neighborhood_slug FROM listings\'+where+\n                (\' AND \' if where else \' WHERE \')+\'neighborhood_slug IS NOT NULL ORDER BY neighborhood_slug\', params)] if city else []\n            types = [dict(label=k,count=db.execute(\'SELECT count(*) FROM listings\'+where+(\' AND \' if where else \' WHERE \')+\'cat3_slug=?\',params+(k,)).fetchone()[0]) for k in categories]\n            return dict(count=n,average_price=avg,median_price=median(\'price_value\'),median_price_per_sqm=median(\'price_per_sqm\'),\n                cities_count=len(city_list),available_cities=city_list,neighborhoods=neighborhoods,\n                distributions=dict(price=histogram(\'price_value\'),area=histogram(\'building_size\'),year=histogram(\'construction_year\'),\n                    property_types=types,amenities=amenities,\n                    missing_years=db.execute("SELECT count(*) FROM listings"+where+(\' AND \' if where else \' WHERE \')+"json_extract(payload,\'$.construction_year\') IS NULL",params).fetchone()[0],\n                    price_area=[dict(id=p[\'id\'],city=p[\'city_slug\'],area=p[\'building_size\'],price=p[\'price_value\']) for p in sample]))\n\n        groups = {}\n        for city in [None]+cities:\n            for category in [None]+categories:\n                groups[group_key(city,category)] = aggregate(city,category)\n        progress(f\'Built exact aggregate statistics for {len(cities)} represented cities\')\n        # Fixed schema permits an entirely missing amenity/year column in a chunk.\n        from .schemas import Property\n        schema = pa.schema([(name, pa.string() if name in (\'id\',\'city_slug\',\'cat3_slug\',\'neighborhood_slug\',\'created_at_month\',\'source_identity\',\'source_revision\')\n            else pa.bool_() if name.startswith(\'has_\') else pa.int64() if name==\'source_row_index\' else pa.float64()) for name in Property.model_fields])\n        for city, category in db.execute(\'SELECT DISTINCT city_slug,cat3_slug FROM listings ORDER BY 1,2\').fetchall():\n            cursor = db.execute(\'SELECT payload FROM listings WHERE city_slug=? AND cat3_slug=? ORDER BY neighborhood_slug,id\',(city,category))\n            chunk = 0\n            while rows := cursor.fetchmany(chunk_rows):\n                stem = f\'data/{city}/{category}/{chunk:05d}\'\n                records = save_chunk(rows, stem+\'.json.gz\',\'partition\',city=city,category=category)\n                partitions.append(stem+\'.json.gz\')\n                target = output/(stem+\'.parquet\')\n                pq.write_table(pa.Table.from_pylist(records,schema=schema),target,compression=\'zstd\')\n                files[stem+\'.parquet\'] = dict(sha256=file_hash(target),bytes=target.stat().st_size,kind=\'parquet\',rows=len(records),city=city,category=category)\n                files[stem+\'.json.gz\'][\'neighborhoods\'] = sorted({p[\'neighborhood_slug\'] for p in records if p.get(\'neighborhood_slug\')})\n                for p in records:\n                    db.execute(\'INSERT INTO lookup VALUES (?,?)\',(p[\'id\'],stem+\'.json.gz\'))\n                chunk += 1\n        for prefix in (f\'{i:02x}\' for i in range(256)):\n            mapping = dict(db.execute(\'SELECT id,path FROM lookup WHERE id>=? AND id<?\',(prefix,prefix+\'g\')))\n            if mapping:\n                save(\'lookup/\'+prefix+\'.json.gz\',gzip.compress(encode(mapping),mtime=0),\'lookup\')\n        views, city_views = {}, {}\n        for city in [None]+cities:\n            current={}\n            for sort in SORTS:\n                paths = []\n                where,params=clause(city,None)\n                cursor = db.execute(f\'SELECT payload FROM listings{where} ORDER BY {sort},id\',params)\n                chunk = 0\n                while rows := cursor.fetchmany(chunk_rows):\n                    path = f\'views/{"city-"+city if city else "global"}/{sort}/{chunk:05d}.json.gz\'\n                    save_chunk(rows,path,\'view\',sort=sort,city=city)\n                    paths.append(path)\n                    chunk += 1\n                current[sort]=paths\n            if city:\n                city_views[city]=current\n            else:\n                views=current\n        periods = db.execute("SELECT min(json_extract(payload,\'$.created_at_month\')), max(json_extract(payload,\'$.created_at_month\')) FROM listings").fetchone()\n        save(\'catalog.json\',encode(dict(cities=cities,groups=groups,partitions=partitions,views=views,city_views=city_views,\n                                       period_start=periods[0],period_end=periods[1])),\'catalog\')\n        manifest = dict(schema_version=SCHEMA_VERSION,source_repo=SOURCE_REPO,source_revision=SOURCE_REVISION,license=\'odbl-1.0\',\n            created_at=datetime.now(timezone.utc).isoformat(),total_source_rows=counts[\'source_rows\'],\n            expected_source_rows=sum(p[\'expected_rows\'] for p in source_parts),total_prepared_rows=total,total_cities=len(cities),\n            full_source_verified=True,source_files=inventory,counts=dict(counts),files=files,\n            duplicate_policy=\'Identical normalized safe fields, including city, neighborhood and advertisement month; may merge different homes.\',\n            source_scope=\'All CSV/Parquet data files in the pinned source repository inventory; not active listings or the entire housing market.\')\n        validate_manifest(manifest)\n        (output/\'manifest.json\').write_bytes(encode(manifest))\n        validate_output(output)\n        progress(f\'FINISHED: {counts["source_rows"]:,} source rows; {total:,} prepared sales; {len(cities)} cities\')\n        return manifest\n    finally:\n        db.close()  # Keep staging file for diagnostics/recovery; it is not uploaded.\n\n\ndef validate_output(output):\n    """Re-read every output, checking file integrity, privacy, types and record counts."""\n    import pyarrow.parquet as pq\n    output = Path(output)\n    if (output/\'manifest.json\').stat().st_size>8*1024*1024:\n        raise ValueError(\'Manifest exceeds the 8 MiB serverless serving guard\')\n    manifest = validate_manifest(json.loads((output/\'manifest.json\').read_bytes()))\n    rows = 0\n    for path, info in manifest[\'files\'].items():\n        target = output/path\n        if target.stat().st_size != info[\'bytes\'] or file_hash(target) != info[\'sha256\']:\n            raise ValueError(\'Generated file failed integrity validation\')\n        if info[\'kind\'] in (\'partition\',\'view\'):\n            records = json.loads(gzip.decompress(target.read_bytes()))\n            if len(records) != info[\'rows\']:\n                raise ValueError(\'Prepared row count mismatch\')\n            for record in records:\n                validate_record(record)\n            if info[\'kind\']==\'partition\':\n                rows += len(records)\n        elif info[\'kind\']==\'parquet\':\n            source=pq.ParquetFile(target)\n            if source.metadata.num_rows != info[\'rows\']:\n                raise ValueError(\'Parquet count mismatch\')\n            for batch in source.iter_batches(batch_size=1024):\n                for record in batch.to_pylist():\n                    validate_record(record)\n    if rows != manifest[\'total_prepared_rows\']:\n        raise ValueError(\'Prepared partition inventory is incomplete\')\n    return manifest\n', 'prepared_upload': '"""Explicit Colab-only publication and read-back verification, never HTTP serving."""\nimport json\nfrom pathlib import Path\nimport re\n\nfrom .prepared_format import TARGET_REPO, file_hash, sha256\nfrom .preparation import validate_output\n\nBATCH_FILES = 200\nBATCH_BYTES = 32 * 1024 * 1024\n\nDATASET_CARD = \'\'\'---\nlicense: odbl\nlanguage:\n- fa\ntask_categories:\n- tabular-regression\n---\n# AMLAAAK prepared historical residential sales\n\nDerived from Divar\'s [official real_estate_ads dataset](https://huggingface.co/datasets/divarofficial/real_estate_ads),\nrevision e9446ee97b8e6eb9bd62bdfd7548f1bc8973be85, under the\n[Open Database License 1.0](https://opendatacommons.org/licenses/odbl/1-0/).\nAttribution: Divar / divarofficial. Preserve the original source attribution and\nODbL notices when using or redistributing this derived database. Private storage\ndoes not remove license obligations. Review the source card and license before sharing.\n\nTransformations: structured-field allowlist, numeral/numeric/boolean/year parsing,\nfixed-price apartment/house-villa sale eligibility, rental exclusion, and exact\nnormalized-safe-field deduplication (which can merge distinct homes).\nNo descriptions, titles, contacts, advertisement tokens, listing URLs or exact\ncoordinates are included. Geographic fields are public city/neighborhood slugs.\nHistorical asking prices retain original source units; currency is unverified.\nRecords are not active advertisements. This archive is not new model training data.\n\n`prepared/manifest.json` describes verified source partitions, counts, schema,\nfile SHA-256 hashes and processing coverage. Parquet is the research interchange;\nchecksummed gzip JSON chunks and indexes support bounded serverless serving.\nThe raw source and temporary staging database are not uploaded.\n\'\'\'\n\n\ndef upload_prepared(output, api, *, confirmation, license_acknowledged, repo=TARGET_REPO):\n    """No remote writes before exact confirmation, privacy/integrity and private-repo checks.\n\n    Caller must display the file/record summary and proposed README before asking.\n    Large uploads use bounded commits; manifest/card are published last. No\n    repository creation, visibility change, deletions, or model uploads occur.\n    """\n    if confirmation != \'UPLOAD PREPARED DATASET\' or license_acknowledged is not True:\n        raise PermissionError(\'Upload not approved; no remote files were changed\')\n    if repo != TARGET_REPO:\n        raise ValueError(\'Target dataset identity differs from the approved private repository\')\n    root=Path(output)\n    manifest=validate_output(root)\n    if not manifest[\'full_source_verified\']:\n        raise ValueError(\'Full pinned-source processing has not been verified; refusing upload\')\n    info=api.dataset_info(repo_id=repo)\n    if info.id != repo or info.private is not True:\n        raise ValueError(\'Upload requires the existing approved PRIVATE dataset repository\')\n    from huggingface_hub import CommitOperationAdd\n    paths=sorted(manifest[\'files\'])+[\'manifest.json\']\n    parent=info.sha\n    def commit(operations):\n        nonlocal parent\n        result=api.create_commit(repo_id=repo,repo_type=\'dataset\',operations=operations,\n            commit_message=\'Prepare verified historical Divar sales archive\',parent_commit=parent)\n        if not re.fullmatch(r\'[a-f0-9]{40}\',result.oid):\n            raise ValueError(\'Upload returned an invalid immutable revision\')\n        parent=result.oid\n        print(f\'Committed {len(operations)} prepared file operation(s); revision {parent}\')\n    operations=[]\n    size=0\n    for path in sorted(manifest[\'files\']):\n        current=manifest[\'files\'][path][\'bytes\']\n        if operations and (len(operations)>=BATCH_FILES or size+current>BATCH_BYTES):\n            commit(operations)\n            operations=[]\n            size=0\n        operations.append(CommitOperationAdd(path_in_repo=\'prepared/\'+path,path_or_fileobj=str(root/path)))\n        size+=current\n    final=[CommitOperationAdd(path_in_repo=\'prepared/manifest.json\',path_or_fileobj=str(root/\'manifest.json\')),\n           CommitOperationAdd(path_in_repo=\'README.md\',path_or_fileobj=DATASET_CARD.encode())]\n    if operations and (len(operations)+2>BATCH_FILES or size+(root/\'manifest.json\').stat().st_size+len(DATASET_CARD.encode())>BATCH_BYTES):\n        commit(operations)\n        operations=[]\n    commit(operations+final)  # Only this last commit publishes the new ready manifest.\n    revision=parent\n    remote=api.dataset_info(repo_id=repo,revision=revision)\n    if remote.id != repo or remote.private is not True or remote.sha != revision:\n        raise ValueError(\'Uploaded repository identity/revision verification failed\')\n    uploaded=set(api.list_repo_files(repo_id=repo,repo_type=\'dataset\',revision=revision))\n    required={\'prepared/\'+p for p in paths}|{\'README.md\'}\n    if not required <= uploaded:\n        raise ValueError(\'Uploaded file inventory is incomplete\')\n    # Download from the immutable commit to a new verification cache. Do not\n    # accept the local upload source as proof of remote integrity.\n    from huggingface_hub import hf_hub_download\n    import tempfile\n    cache=tempfile.mkdtemp(prefix=\'amlaaak-upload-verify-\')\n    for index,path in enumerate(paths,1):\n        fetched=hf_hub_download(repo_id=repo,repo_type=\'dataset\',revision=revision,\n            filename=\'prepared/\'+path,token=api.token,cache_dir=cache)\n        expected=file_hash(root/\'manifest.json\') if path==\'manifest.json\' else manifest[\'files\'][path][\'sha256\']\n        if file_hash(fetched)!=expected:\n            raise ValueError(\'Remote read-back integrity verification failed\')\n        if index%100==0 or index==len(paths):\n            print(f\'Remote integrity verified: {index}/{len(paths)} files\')\n    card=hf_hub_download(repo_id=repo,repo_type=\'dataset\',revision=revision,\n        filename=\'README.md\',token=api.token,cache_dir=cache)\n    if file_hash(card)!=sha256(DATASET_CARD.encode()):\n        raise ValueError(\'Remote attribution card integrity verification failed\')\n    result=dict(DATASET_REPO=repo,DATASET_REVISION=revision,\n        DATASET_MANIFEST_SHA256=file_hash(root/\'manifest.json\'),TOTAL_SOURCE_ROWS=manifest[\'total_source_rows\'],\n        TOTAL_PREPARED_ROWS=manifest[\'total_prepared_rows\'],TOTAL_CITIES=manifest[\'total_cities\'])\n    print(\'\\nVERIFIED UPLOAD — COPY THESE VALUES\\n\'+json.dumps(result,indent=2))\n    return result\n'}
for name, source in EMBEDDED_SOURCES.items():
    (PACKAGE/(name+'.py')).write_text(source,encoding='utf-8')
sys.path.insert(0,str(WORK))
from amlaaak_colab.prepared_format import SOURCE_REPO, SOURCE_REVISION, TARGET_REPO, file_hash
from amlaaak_colab.preparation import build_dataset, validate_output, inspect_source
from amlaaak_colab.prepared_upload import upload_prepared, DATASET_CARD


## 1. Check the Colab runtime and pinned source inventory


In [ ]:
import os, shutil
import pyarrow as pa
from huggingface_hub import HfApi, hf_hub_download, DatasetCard
from datetime import datetime, timezone
assert Path('/content').exists(), 'Use Google Colab for full-source processing.'
free=shutil.disk_usage('/content').free
ram=os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES')
print(f'RAM: {ram/2**30:.1f} GiB; free disk: {free/2**30:.1f} GiB; CPU count: {os.cpu_count()}')
assert free>8*2**30, 'At least 8 GiB free Colab disk is required; choose a fresh runtime.'
assert ram>3*2**30, 'Choose a Colab runtime with at least 3 GiB RAM.'
pa.set_cpu_count(2)
public_api=HfApi(token=False)
info=public_api.dataset_info(repo_id=SOURCE_REPO,revision=SOURCE_REVISION,files_metadata=True)
assert info.sha==SOURCE_REVISION, 'Pinned revision did not resolve exactly.'
source_card_path=hf_hub_download(repo_id=SOURCE_REPO,repo_type='dataset',revision=SOURCE_REVISION,filename='README.md',token=False)
source_card=DatasetCard.load(source_card_path)
assert source_card.data.to_dict().get('license') in ('odbl','odbl-1.0'), 'Unexpected source license; stop and review.'
print('Source attribution: Divar / divarofficial; license ODbL 1.0')
print('Review: https://huggingface.co/datasets/divarofficial/real_estate_ads')
print('License: https://opendatacommons.org/licenses/odbl/1-0/')
tree=list(public_api.list_repo_tree(repo_id=SOURCE_REPO,repo_type='dataset',revision=SOURCE_REVISION,recursive=True))
partitions=sorted([f for f in tree if getattr(f,'path','').endswith(('.csv','.parquet'))],key=lambda f:f.path)
expected_names={f.rfilename for f in info.siblings if f.rfilename.endswith(('.csv','.parquet'))}
assert partitions and {f.path for f in partitions}==expected_names, 'Source inventories disagree or are empty.'
assert all(getattr(f,'lfs',None) and f.lfs.sha256 for f in partitions), 'Every source partition must have an independently pinned LFS SHA256.'
print('EXACT SOURCE REVISION:',SOURCE_REVISION)
print('Expected source partitions:',len(partitions))
for f in partitions:
    print(f.path, f.size, 'bytes; SHA256',f.lfs.sha256)


## 2. Read all source partitions directly into Colab

Downloads stay in the temporary Colab runtime. Only the structured allowlist is
read into batches; raw descriptions/contacts are never copied into prepared files.
Original source files remain only in the Colab cache and are **not uploaded**.


In [ ]:
import pyarrow.parquet as pq
from tqdm.auto import tqdm
source_parts=[]
for part in tqdm(partitions,desc='Pinned source files'):
    local=hf_hub_download(repo_id=SOURCE_REPO,repo_type='dataset',revision=SOURCE_REVISION,
        filename=part.path,token=False,cache_dir=str(WORK/'source-cache'))
    assert Path(local).stat().st_size==part.size, 'Source size mismatch'
    assert file_hash(local)==part.lfs.sha256, 'Source integrity mismatch'
    source_parts.append(dict(path=local,source_path=part.path,sha256=part.lfs.sha256,
                             expected_rows=inspect_source(local)))
assert {p['source_path'] for p in source_parts}==expected_names, 'Missing source partitions'
print('TOTAL EXPECTED SOURCE ROWS:',sum(p['expected_rows'] for p in source_parts))
print('All pinned source files are present and hash-verified in Colab.')


## 3. Prepare, index, validate and count

Uses batches of 1,024 source records and output chunks of at most 2,000 records.
A temporary on-disk SQLite staging file performs deduplication and exact aggregate
calculations. It is not a hosted database and is never uploaded. The application
receives Parquet plus compact indexed gzip JSON serving files, including global
sorted page indexes, exact city/category statistics, and neighborhood indexes.
Deduplication compares normalized safe fields (including month and neighborhood);
it may merge different homes with identical recorded specifications. The manifest
reports this limitation and all exclusions. All represented valid source slugs
are handled without a hardcoded city allowlist; invalid slugs are reported.


In [ ]:
run_id=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=WORK/('prepared-'+run_id)
manifest=build_dataset(source_parts,OUTPUT)
manifest=validate_output(OUTPUT)
assert manifest['full_source_verified']
assert manifest['total_source_rows']==sum(p['expected_rows'] for p in source_parts)
assert {p['source_path'] for p in manifest['source_files']}==expected_names
print(json.dumps({k:manifest[k] for k in ['total_source_rows','expected_source_rows','total_prepared_rows','total_cities','full_source_verified','counts']},indent=2))
print('MANIFEST SHA256:',file_hash(OUTPUT/'manifest.json'))
print('Output files:',len(manifest['files'])+1)
print('Output size (MiB):',round(sum(p.stat().st_size for p in OUTPUT.rglob('*') if p.is_file())/2**20,2))
print('Privacy validation passed: no raw titles, descriptions, contact details, exact coordinates or advertisement tokens.')
print('No upload has occurred. Current model remains unchanged.')


## UPLOAD PREPARED DATASET TO HUGGING FACE

Review the counts above. Continue **only** if every source partition completed
and validation succeeded. Review the attribution card below and the source
[ODbL 1.0 license](https://opendatacommons.org/licenses/odbl/1-0/).

This final cell will ask you to acknowledge the license, then type
**UPLOAD PREPARED DATASET**. Any other answer stops before upload. It uploads only
the prepared files plus an attribution README, replacing paths with the same names
in the existing private repository; it does not delete remote files, publish the
repository, upload raw/staging files, or touch the model.

For authentication, add `HF_TOKEN` in Colab's **Secrets** (key icon), granting this
notebook access. Use a fine-grained write token for this dataset. If no Secret is
available, a hidden token prompt appears. Tokens are never printed or embedded.
Vercel later needs a separate **read-only** token on the backend only.


In [ ]:
print(DATASET_CARD)
print('Target existing PRIVATE dataset:',TARGET_REPO)
print('Upload file count:',len(manifest['files'])+2,'including attribution README and manifest')
print('Prepared records:',manifest['total_prepared_rows'],'Cities:',manifest['total_cities'])
license_ack=input('Type I ACCEPT ODbL 1.0 after reviewing source license and attribution: ').strip()
confirmation=input('Type UPLOAD PREPARED DATASET to approve, or press Enter to stop: ').strip()
if license_ack!='I ACCEPT ODbL 1.0' or confirmation!='UPLOAD PREPARED DATASET':
    raise PermissionError('Upload not approved. No remote files were changed.')
token=None
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
except Exception:
    pass
if not token:
    from getpass import getpass
    token=getpass('Hugging Face dataset WRITE token (hidden): ').strip()
assert token, 'No token provided; no upload.'
private_api=HfApi(token=token)
result=upload_prepared(OUTPUT,private_api,confirmation=confirmation,license_acknowledged=True)
# Output is shown only AFTER every remote file is verified at the returned commit.
print('BACKEND VERCEL ENVIRONMENT VARIABLES — apps/api project only:')
print('AMLAAAK_DATA_MODE=prepared')
print('AMLAAAK_PREPARED_DATA_REPO='+result['DATASET_REPO'])
print('AMLAAAK_PREPARED_DATA_REVISION='+result['DATASET_REVISION'])
print('AMLAAAK_PREPARED_DATA_MANIFEST_SHA256='+result['DATASET_MANIFEST_SHA256'])
print('HF_TOKEN=<a separate private-dataset READ token; never paste it into notebook output>')
print('Keep all approved MODEL variables unchanged. Keep frontend API_BASE_URL unchanged.')


## Finished

Copy the verified output values into the **backend** Vercel project's environment
variables, add a read-only token there, and redeploy only after the reviewed code
has been pushed. Do not change the frontend API_BASE_URL or approved model settings.
Run the hosted checks in `docs/PREPARED_DATA_QUICKSTART.md`. More searchable records
do not improve or retrain the existing model. If any validation fails, do not switch
production; the original `huggingface` provider remains available for rollback.
